# Exercise 2 - ReACT code generation

Tools actually used: ChatGPT for prompt design, recorded AI responses, critique, and Python code generation; Python 3 for local execution. The notebooks use only the Python standard library and are designed to run in Google Colab.

## Full user prompt

Use a ReACT-style cycle to generate and test a Python function named summarize_sales(values).
PLAN: Give a short implementation plan, including input validation and edge cases.
ACT: Generate code using only the Python standard library. Input must be a list or tuple of finite, nonnegative int/float values; reject booleans. Return a dictionary with count, total, and average. Round total and average to two decimal places. For empty input return count 0, total 0.0, and average 0.0. Raise TypeError for invalid container types or nonnumeric elements; raise ValueError for negative or nonfinite values, or a nonfinite total.
RUN: Execute normal and empty cases, zero, a tuple, a string container, a string element, a boolean, a negative number, NaN, infinity, and an overflowing total. Print actual results and test outcomes.
OBSERVE: Report the actual test result; do not claim success without execution.
FIX: If a test fails, revise the code and rerun the checks. Keep PLAN, ACT, RUN, OBSERVE, and FIX visibly separate. Provide brief decision summaries, not private internal reasoning.

No separate exercise-specific system prompt was used.

## PLAN

Validate the container and each sale, calculate total and average, and return a dictionary. An empty list needs a special case. Nonfinite numbers and overflow must be rejected.

## ACT / RUN: first draft

This deliberately minimal first draft demonstrates a real empty-input failure. The expected exception is caught so Run all can continue.

In [1]:
def summarize_sales_v1(values):
    return {'count': len(values), 'total': round(sum(values), 2),
            'average': round(sum(values) / len(values), 2)}
print('Normal case:', summarize_sales_v1([19.99, 5.01, 10.00]))
try:
    summarize_sales_v1([])
except ZeroDivisionError as error:
    print('OBSERVE: empty-input test failed:', type(error).__name__)

Normal case: {'count': 3, 'total': 35.0, 'average': 11.67}
OBSERVE: empty-input test failed: ZeroDivisionError


## OBSERVE / FIX

The normal case passed, but the empty-input run raised ZeroDivisionError. Add an empty-input branch, explicit type checks, finite/nonnegative checks, and overflow handling. Booleans need a separate check because Python treats bool as an int subclass.

In [2]:
import math

def summarize_sales(values):
    if not isinstance(values, (list, tuple)):
        raise TypeError('values must be a list or tuple')
    numbers = []
    for value in values:
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError('each sale must be an int or float, not a boolean')
        try:
            number = float(value)
        except OverflowError:
            raise ValueError('sale is too large') from None
        if not math.isfinite(number) or number < 0:
            raise ValueError('each sale must be finite and nonnegative')
        numbers.append(number)
    try:
        total = math.fsum(numbers)
    except OverflowError:
        raise ValueError('total is too large') from None
    if not math.isfinite(total):
        raise ValueError('total must be finite')
    count = len(numbers)
    return {'count': count, 'total': round(total, 2),
            'average': round(total / count, 2) if count else 0.0}

## RUN: revised code

In [3]:
valid_cases = [
    ('normal', [19.99, 5.01, 10.0], {'count': 3, 'total': 35.0, 'average': 11.67}),
    ('empty', [], {'count': 0, 'total': 0.0, 'average': 0.0}),
    ('zero', [0], {'count': 1, 'total': 0.0, 'average': 0.0}),
    ('tuple', (10, 20), {'count': 2, 'total': 30.0, 'average': 15.0}),
]
for label, values, expected in valid_cases:
    actual = summarize_sales(values)
    assert actual == expected, (label, actual, expected)
    print(f'PASS {label}: {actual}')
invalid_cases = [
    ('string container', '10,20', TypeError),
    ('string element', [10, '20'], TypeError),
    ('boolean', [True], TypeError),
    ('negative', [-1], ValueError),
    ('NaN', [float('nan')], ValueError),
    ('infinity', [float('inf')], ValueError),
    ('overflowing total', [1e308, 1e308], ValueError),
]
for label, values, expected_error in invalid_cases:
    try:
        summarize_sales(values)
    except expected_error as error:
        print(f'PASS {label}: {type(error).__name__}: {error}')
    else:
        raise AssertionError(f'{label}: expected {expected_error.__name__}')
print('All 11 checks passed.')

PASS normal: {'count': 3, 'total': 35.0, 'average': 11.67}
PASS empty: {'count': 0, 'total': 0.0, 'average': 0.0}
PASS zero: {'count': 1, 'total': 0.0, 'average': 0.0}
PASS tuple: {'count': 2, 'total': 30.0, 'average': 15.0}
PASS string container: TypeError: values must be a list or tuple
PASS string element: TypeError: each sale must be an int or float, not a boolean
PASS boolean: TypeError: each sale must be an int or float, not a boolean
PASS negative: ValueError: each sale must be finite and nonnegative
PASS NaN: ValueError: each sale must be finite and nonnegative
PASS infinity: ValueError: each sale must be finite and nonnegative
PASS overflowing total: ValueError: total is too large
All 11 checks passed.


## OBSERVE

All 11 checks passed in local Python execution. The notebook retains these outputs. Run it in Colab before submitting; the current evidence is not a claim of a completed Colab session.